In [12]:
# Notebook parameters (modify as needed)#
#model_name = "facebook/opt-2.7b"
model_name = "huggyllama/llama-7b"
prompt = "I've always wondered what"
m = 80            # length of generation
n = 256           # watermark sequence length
key = 42          # watermark key
seed = 1          # random seed for reproducibility

In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device(
    'cuda' if torch.cuda.is_available() else
    ('mps' if torch.backends.mps.is_available() else 'cpu')
)
print(device)

cpu


In [14]:
# Seed for reproducibility
torch.manual_seed(seed)

# Initialize tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
)

# Prepare inputs
inputs = tokenizer.encode(prompt, return_tensors='pt').to(device)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [15]:
# Generate normally
outputs = model.generate(
    inputs,
    max_new_tokens=m,
    do_sample=True,
)

# Decode and display
normal_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("--- Normal Output ---")
print(normal_text)

--- Normal Output ---
I've always wondered what it's like to be young(er) and have a blog. I mean you've got lots of time, it's just now that you have to work a 9 to 5 job, that you feel you have no time. And there's also the feeling of self-consciousness that plagues anyone who starts to blog. But I'm sure once you


In [16]:
# Mersenne Twister based RNG for watermarking
def int_32(x): return int(0xFFFFFFFF & x)

class mersenne_rng:
    def __init__(self, seed=5489):
        self.state = [0]*624
        self.index = 624
        self.upper_mask = (1<<31)
        self.lower_mask = (1<<31) - 1
        # constants
        self.f, self.m = 1812433253, 397
        self.u, self.s, self.b = 11, 7, 0x9D2C5680
        self.t, self.c, self.l = 15, 0xEFC60000, 18
        # initialize state
        self.state[0] = seed
        for i in range(1, 624):
            prev = self.state[i-1]
            self.state[i] = int_32(self.f * (prev ^ (prev >> 30)) + i)

    def twist(self):
        for i in range(624):
            x = (self.state[i] & self.upper_mask) + (self.state[(i+1)%624] & self.lower_mask)
            xA = x >> 1
            if x % 2: xA ^= 0x9908b0df
            self.state[i] = self.state[(i + self.m) % 624] ^ xA
        self.index = 0

    def rand(self):
        if self.index >= 624:
            self.twist()
        y = self.state[self.index]
        y ^= (y >> self.u)
        y ^= ((y << self.s) & self.b)
        y ^= ((y << self.t) & self.c)
        y ^= (y >> self.l)
        self.index += 1
        return (int_32(y) * (1.0 / 4294967296.0))

# Exponential watermark sampling
def exp_sampling(probs, u):
    # probs: tensor [vocab], u: numpy array [vocab]
    return torch.argmax(torch.from_numpy(u) ** (1.0 / probs.cpu().numpy()), dim=1).unsqueeze(-1)

# Watermark generation
import numpy as np
import tqdm

def generate_shift(model, prompt_ids, vocab_size, n, m, key):
    rng = mersenne_rng(key)
    # create watermark matrix xi
    xi = np.array([rng.rand() for _ in range(n * vocab_size)], dtype=np.float32).reshape(n, vocab_size)
    shift = torch.randint(0, n, (1,)).item()

    inputs = prompt_ids.to(model.device)
    attn = torch.ones_like(inputs)
    past = None
    for i in tqdm.tqdm(range(m)):
        with torch.no_grad():
            if past:
                out = model(inputs[:, -1:], past_key_values=past, attention_mask=attn)
            else:
                out = model(inputs)
        probs = torch.nn.functional.softmax(out.logits[:, -1, :vocab_size], dim=-1)
        u = xi[(shift + i) % n]
        token = exp_sampling(probs, u)
        inputs = torch.cat([inputs, token.to(model.device)], dim=-1)
        past = out.past_key_values
        attn = torch.cat([attn, torch.ones((attn.shape[0],1), device=attn.device)], dim=-1)

    return inputs.cpu()

In [17]:
# Encode prompt & run watermark generation
tokens = tokenizer.encode(prompt, return_tensors='pt')
wm_tokens = generate_shift(model, tokens, len(tokenizer), n, m, key)[0]
watermarked_text = tokenizer.decode(wm_tokens, skip_special_tokens=True)
print()
print("--- Watermarked Output ---")
print(watermarked_text)

  0%|          | 0/80 [00:00<?, ?it/s]<ipython-input-16-6c860dbcbbc3>:42: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  return torch.argmax(torch.from_numpy(u) ** (1.0 / probs.cpu().numpy()), dim=1).unsqueeze(-1)
100%|██████████| 80/80 [02:55<00:00,  2.19s/it]


--- Watermarked Output ---
I've always wondered what mages and priests have in common. Well, we seem to be registering fire now (or try to) which is one of the things priests can do even if they aren't at their full holy power. I also noticed that my channeling closes the door to fire elementals while all my magery is open to elementals who take the same damage deal as others.


In [18]:
from numba import njit, prange
import numpy as np
import math

@njit
def levenshtein(x, y, gamma=0.0):
    n = len(x)
    m = len(y)
    A = np.zeros((n + 1, m + 1), dtype=np.float64)
    for i in range(n + 1):
        for j in range(m + 1):
            if i == 0:
                A[i, j] = j * gamma
            elif j == 0:
                A[i, j] = i * gamma
            else:
                cost_sub = math.log(1.0 - y[j - 1, x[i - 1]])
                # deletion
                best = A[i - 1, j] + gamma
                # insertion
                tmp = A[i, j - 1] + gamma
                if tmp < best:
                    best = tmp
                # substitution
                tmp = A[i - 1, j - 1] + cost_sub
                if tmp < best:
                    best = tmp
                A[i, j] = best
    return A[n, m]


In [19]:
import numpy as np
import time
import tqdm

from numba import njit, prange


@njit
def detect(tokens, n, k, xi, gamma=0.0):
    m = len(tokens)
    n = len(xi)

    A = np.empty((m - (k - 1), n))
    for i in range(m - (k - 1)):
        for j in range(n):
            A[i][j] = levenshtein(tokens[i : i + k], xi[(j + np.arange(k)) % n], gamma)

    return np.min(A)

def permutation_test(tokens, key, n, k, vocab_size, n_runs=100):
    rng = mersenne_rng(key)
    xi = np.array(
        [rng.rand() for _ in range(n * vocab_size)], dtype=np.float32
    ).reshape(n, vocab_size)
    test_result = detect(tokens, n, k, xi)

    p_val = 0
    for run in tqdm.tqdm(range(n_runs)):
        xi_alternative = np.random.rand(n, vocab_size).astype(np.float32)
        null_result = detect(tokens, n, k, xi_alternative)

        # assuming lower test values indicate presence of watermark
        p_val += null_result <= test_result

    return (p_val + 1.0) / (n_runs + 1.0)


# Run decoding test
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokens = tokenizer.encode(watermarked_text, return_tensors='pt', truncation=True, max_length=2048).numpy()[0]
tok_list = tokens.tolist()
print(tok_list)
start = time.time()
p_val_watermarked = permutation_test(tok_list, key, n, len(tok_list), len(tokenizer))
print()
print(f"p-value: {p_val_watermarked:.4f} (elapsed {time.time()-start:.1f}s)")
print(f"Is watermarked: {p_val_watermarked<0.01}")

[1, 306, 29915, 345, 2337, 4997, 287, 825, 286, 1179, 322, 3691, 9197, 505, 297, 3619, 29889, 5674, 29892, 591, 2833, 304, 367, 6036, 292, 3974, 1286, 313, 272, 1018, 304, 29897, 607, 338, 697, 310, 278, 2712, 3691, 9197, 508, 437, 1584, 565, 896, 9455, 29915, 29873, 472, 1009, 2989, 26630, 3081, 29889, 306, 884, 10548, 393, 590, 8242, 292, 4694, 267, 278, 3050, 304, 3974, 1543, 1338, 1550, 599, 590, 2320, 708, 338, 1722, 304, 1543, 1338, 1058, 2125, 278, 1021, 18658, 5376, 408, 4045, 29889]


100%|██████████| 100/100 [00:27<00:00,  3.70it/s]


p-value: 0.0099 (elapsed 43.1s)
Is watermarked: True


In [20]:
!pip install googletrans==4.0.0-rc1
from googletrans import Translator
import random

def back_translate(text, target_language='es'):
    """Translates text to a target language and back to the original language."""
    translator = Translator()
    translated = translator.translate(text, dest=target_language).text
    back_translated = translator.translate(translated, dest='en').text
    return back_translated

def add_noise(text, noise_level=0.1):
    """Adds random noise to the text."""
    words = text.split()
    num_noise_words = int(len(words) * noise_level)
    noise_words = ["noise" for _ in range(num_noise_words)]
    for i in range(num_noise_words):
        words.insert(random.randint(0, len(words)), noise_words[i])
    return " ".join(words)

# Apply back-translation to the watermarked text
back_translated_text = back_translate(back_translate(watermarked_text, "fr"), "ru")[:300]
print("Back translated: ", back_translated_text)

# Encode the back-translated text
back_translated_tokens = tokenizer.encode(back_translated_text, return_tensors='pt', truncation=True, max_length=2048).numpy()[0]
back_translated_tok_list = back_translated_tokens.tolist()

# Detect watermark in the back-translated text
start = time.time()
p_val_back_translated = permutation_test(back_translated_tok_list, key, n, len(back_translated_tok_list), len(tokenizer))
print()
print(f"p-value (back-translated): {p_val_back_translated:.4f} (elapsed {time.time()-start:.1f}s)")
print(f"Is watermarked (back-translated): {p_val_back_translated<0.01}")

Back translated:  I always wondered what common magicians and priests. Well, now we seem to record fire (or try), what is one of the things that priests can do, even if they are not in full power. I also noticed that my pipe strengthens the door so that fires, while all my magic is open for elementary people who caus


100%|██████████| 100/100 [00:14<00:00,  6.72it/s]


p-value (back-translated): 0.0099 (elapsed 22.1s)
Is watermarked (back-translated): True


In [21]:
control_text = "This essay was written by a human I swear. There is nothing watermarked here and nothing suspicious to see. I worked very hard to produce this text."
tokens_control = tokenizer.encode(control_text, return_tensors='pt', truncation=True, max_length=2048).numpy()[0]
tok_control_list = tokens_control.tolist()
print(tok_control_list)
start = time.time()
p_val_control = permutation_test(tok_control_list, key, n, len(tok_control_list), len(tokenizer))
print()
print(f"p-value (control): {p_val_control:.4f} (elapsed {time.time()-start:.1f}s)")
print(f"Is watermarked (control): {p_val_control<0.01}")

[1, 910, 3686, 388, 471, 3971, 491, 263, 5199, 306, 7901, 279, 29889, 1670, 338, 3078, 16699, 837, 935, 287, 1244, 322, 3078, 8872, 14803, 304, 1074, 29889, 306, 3796, 1407, 2898, 304, 7738, 445, 1426, 29889]


100%|██████████| 100/100 [00:09<00:00, 10.41it/s]


p-value (control): 0.7525 (elapsed 17.8s)
Is watermarked (control): False


In [22]:
print(f"""
--- Plain text ---
{normal_text}

--- Watermarked (p={p_val_watermarked:.4f}) ---
{watermarked_text}

--- Edited (p={p_val_back_translated:.4f}) ---
{back_translated_text}

--- Control (p={p_val_control:.4f}) ---
{control_text}
""")


--- Plain text ---
I've always wondered what it's like to be young(er) and have a blog. I mean you've got lots of time, it's just now that you have to work a 9 to 5 job, that you feel you have no time. And there's also the feeling of self-consciousness that plagues anyone who starts to blog. But I'm sure once you

--- Watermarked (p=0.0099) ---
I've always wondered what mages and priests have in common. Well, we seem to be registering fire now (or try to) which is one of the things priests can do even if they aren't at their full holy power. I also noticed that my channeling closes the door to fire elementals while all my magery is open to elementals who take the same damage deal as others.

--- Edited (p=0.0099) ---
I always wondered what common magicians and priests. Well, now we seem to record fire (or try), what is one of the things that priests can do, even if they are not in full power. I also noticed that my pipe strengthens the door so that fires, while all my magic is open fo